<a href="https://colab.research.google.com/github/Rogerio-mack/IMT_CD_2026/blob/main/IMT_CD_Exercicio_pipeline_selecao_modelos_e_atributos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exercício: Seleção de Modelos e Atributos**

# Exemplo: `Pipeline`

Veja mais em https://inria.github.io/scikit-learn-mooc/.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# 1. Carregando o dataset Iris
iris = load_iris()
X, y = iris.data, iris.target

# Separação 20% dos dados para o teste final
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Criando o Pipeline estrutural com placeholders ('passthrough')
pipeline = Pipeline([
    ('scaler', 'passthrough'),
    ('classifier', 'passthrough')
])

# 3. Configurando a grade de busca com pré-processamento dinâmico
param_grid = [
    # Família 1: Árvore de Decisão (Não precisa de escala)
    {
        'scaler': ['passthrough'],
        'classifier': [DecisionTreeClassifier(random_state=42)],
        'classifier__max_depth': [3, 5, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    # Família 2: SVC (Exige normalização)
    {
        'scaler': [StandardScaler()],
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    # Família 3: Regressão Logística (Beneficia-se de escala)
    {
        'scaler': [StandardScaler()],
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.1, 1, 10],
        'classifier__solver': ['liblinear']
    }
]

# 4. Executando o GridSearchCV no conjunto de TREINO
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

# Exibindo o melhor resultado encontrado na validação cruzada
print("--- FASE DE TREINAMENTO E VALIDAÇÃO ---")
print(f"Melhores parâmetros: {grid.best_params_}\n")

# 5. Aplicação do pipeline ao conjunto de teste

# O objeto 'grid' se comporta como o melhor modelo retreinado por padrão.
# Ele vai aplicar o scaler (se o campeão exigir) e fazer a predição automaticamente.
y_pred = grid.predict(X_test)

print("--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---")
print(f"Acurácia Final: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação Completo:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


--- FASE DE TREINAMENTO E VALIDAÇÃO ---
Melhores parâmetros: {'classifier': SVC(random_state=42), 'classifier__C': 0.1, 'classifier__kernel': 'linear', 'scaler': StandardScaler()}

--- AVALIAÇÃO FINAL NO CONJUNTO DE TESTE ---
Acurácia Final: 0.9333

Relatório de Classificação Completo:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



# Exercício 1

Empregue os conceitos de `pipeline` acima para fazer a classificação da espécie dos pinguins no dataset a seguir.

1. Você pode excluir dados ausentes
2. Empregue todas as variáveis preditores, fazendo o encode quando necessário
3. Para árvores de decisão e random forest, não empregue hot encode
4. Aplique o scale se necessário ou desejável
5. Verifique além dos modelos já empregados no exemplo, modelos de RandomForest e MLP com diferentes parâmetros (ao menos 3)

In [ ]:
df = sns.load_dataset("penguins")
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


# Exercício 2

Verifique o ganho de informação do exercício anterior e refaça a seleção dos modelos empregando somente os 50% atributos de maior ganho.

**Qual a sua conclusão deste exercício?**